# 50 — DeepEval Agentic Metrics

**What you'll learn:**
- `ToolCorrectnessMetric` — did the agent call the right tools in the right order?
- `TaskCompletionMetric` — did it actually complete the task?
- How to extract tool call history from LangGraph messages
- How to build `LLMTestCase` with `tools_called` and `expected_tools` fields

**Context:** Agent evaluation is fundamentally different from RAG evaluation.
Agents have non-deterministic trajectories and multi-step reasoning chains that
can fail at any point. A single wrong tool call early in a chain can cascade into
a completely wrong final answer — even if the final text looks plausible.

| Challenge | RAG | Agent |
|-----------|-----|-------|
| Non-determinism | Low (same retrieval) | High (tool order varies) |
| Multi-step reasoning | No | Yes — errors compound |
| Tool side effects | No | Yes — tools change state |
| Evaluation unit | Answer | Full trajectory |

In [ ]:
%pip install -q deepeval langchain langchain-openai langgraph python-dotenv

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("API key set.")

In [ ]:
# Why agent eval is harder than RAG eval
# ----------------------------------------
# RAG: retrieve -> generate -> evaluate the answer
# Agent: plan -> tool_call_1 -> observe -> tool_call_2 -> ... -> final answer
#
# The key difference: we need to evaluate the TRAJECTORY, not just the output.
# DeepEval's ToolCorrectnessMetric does exactly this.
#
# Extracting tool calls from LangGraph message history:
# After app.invoke(), result["messages"] contains:
#   HumanMessage   -> the original task
#   AIMessage      -> has .tool_calls = [{"name": "search_web", "args": {...}, "id": "..."}]
#   ToolMessage    -> the tool's response
#   AIMessage      -> final answer (no .tool_calls)


def extract_tool_calls(messages: list) -> tuple:
    """Extract tool names called and final answer from LangGraph message history."""
    tools_called = []
    final_answer = ""
    for msg in messages:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            tools_called.extend(tc["name"] for tc in msg.tool_calls)
        if hasattr(msg, "content") and msg.content and not getattr(msg, "tool_calls", None):
            final_answer = msg.content
    return final_answer, tools_called


print("extract_tool_calls helper defined.")

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent


@tool
def search_web(query: str) -> str:
    """Search the web for factual information."""
    results = {
        "python release year": "Python was first released in 1991 by Guido van Rossum.",
        "eiffel tower height": "The Eiffel Tower is 330 meters tall.",
        "largest planet": "Jupiter is the largest planet in our solar system.",
    }
    for key, val in results.items():
        if key in query.lower():
            return val
    return f"Search result for: {query} — Found relevant information."


@tool
def calculate(expression: str) -> str:
    """Evaluate a simple mathematical expression."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {e}"


@tool
def lookup_fact(topic: str) -> str:
    """Look up a specific fact from a knowledge base."""
    facts = {
        "python": "Python is a high-level, general-purpose programming language.",
        "langgraph": "LangGraph is a library for building stateful multi-actor LLM applications.",
        "openai": "OpenAI was founded in 2015 and created GPT-4 and ChatGPT.",
    }
    return facts.get(topic.lower(), f"No fact found for: {topic}")


AGENT_TASKS = [
    {
        "input": "Search for Python's release year and calculate 2024 minus that year.",
        "expected_tools": ["search_web", "calculate"],
    },
    {"input": "Look up a fact about LangGraph.", "expected_tools": ["lookup_fact"]},
    {
        "input": "What is the height of the Eiffel Tower? Search for it.",
        "expected_tools": ["search_web"],
    },
]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
app = create_react_agent(llm, tools=[search_web, calculate, lookup_fact])

results = []
for task in AGENT_TASKS:
    out = app.invoke({"messages": [{"role": "user", "content": task["input"]}]})
    answer, tools = extract_tool_calls(out["messages"])
    results.append((answer, tools))
    print(f"Task: {task['input'][:60]}")
    print(f"  Tools called:  {tools}")
    print(f"  Expected:      {task['expected_tools']}")
    print(f"  Answer prefix: {answer[:80]}\n")

In [ ]:
from deepeval.metrics import ToolCorrectnessMetric
from deepeval.test_case import ExpectedTool, LLMTestCase, ToolCall

metric = ToolCorrectnessMetric(threshold=0.8, model="gpt-4o-mini")

test_cases = []
for task, (answer, tools_called) in zip(AGENT_TASKS, results):
    tc = LLMTestCase(
        input=task["input"],
        actual_output=answer,
        tools_called=[ToolCall(name=t, input_parameters={}, output="") for t in tools_called],
        expected_tools=[ExpectedTool(name=t) for t in task["expected_tools"]],
    )
    test_cases.append(tc)

for tc in test_cases:
    metric.measure(tc)
    print(f"Input: {tc.input[:60]}")
    print(f"Tools called: {[t.name for t in tc.tools_called]}")
    print(f"Expected:     {[t.name for t in tc.expected_tools]}")
    print(f"Score: {metric.score:.2f} | Pass: {metric.is_successful()}\n")

In [ ]:
# Deliberate failure: mislead the agent into calling the wrong tool
# We ask for a calculation but phrase it as a lookup, so the agent
# might call lookup_fact instead of calculate.

misleading_task = "What does a knowledge base say about 2024 minus 1991?"
out = app.invoke({"messages": [{"role": "user", "content": misleading_task}]})
answer_fail, tools_fail = extract_tool_calls(out["messages"])

print(f"Misleading task: {misleading_task}")
print(f"Tools called:    {tools_fail}")
print("Expected:        ['calculate']")
print(f"Answer:          {answer_fail[:120]}\n")

tc_fail = LLMTestCase(
    input=misleading_task,
    actual_output=answer_fail,
    tools_called=[ToolCall(name=t, input_parameters={}, output="") for t in tools_fail],
    expected_tools=[ExpectedTool(name="calculate")],
)

metric.measure(tc_fail)
print(f"ToolCorrectnessMetric score: {metric.score:.2f}")
print(f"Pass (threshold=0.8):        {metric.is_successful()}")
print("\nIf the agent called the wrong tool, the score should be ~0.0 — FAIL.")

## Exercises

1. **Wrong tool on purpose** — Make the agent call `lookup_fact` on a task that should use `calculate`. Verify `ToolCorrectnessMetric` drops below the 0.8 threshold.

2. **Add a 4th tool** — Implement a `summarize(text: str) -> str` tool and write a task where `expected_tools = ["search_web", "summarize"]`. Does the metric pass when both are called?

3. **End-to-end 5-step task** — Design a task that requires 5 tool calls in sequence (e.g., search -> calculate -> lookup -> calculate -> summarize). Evaluate the full trajectory.

4. **Order sensitivity** — Set `expected_tools` in the reverse order of what the agent calls. Does `ToolCorrectnessMetric` give partial credit, or is it strict about order?

---

**What's next:**
- `49-deepeval-geval-custom` — Build custom G-Eval criteria for domain-specific quality
- `51-deepeval-conversational` — Evaluate multi-turn conversation quality
- `30-agentic-rag` — Combine retrieval with agentic reasoning